In [ ]:
"""Reproduce ENCODE/EpiATLAS 10% cell type overlap error that was present in preprint v1.
"""
# pylint: disable=import-error, redefined-outer-name, too-many-lines
from pathlib import Path

import pandas as pd
from IPython.display import display

from epiclass.utils.notebooks.paper.paper_utilities import ASSAY, CELL_TYPE

Recovers how the ~10% "biospecimen overlap" number in preprint v1 was produced.

The v1 figure was **not** a general biospecimen-source overlap. It was the fraction of ENCODE samples whose cell type falls into one of the **16 cell-type classes used by the Biospecimen classifier**: a narrower definition later misread as overall overlap and generalized to the other classifiers.

Details that are load-bearing and explain why later attempts landed at different values instead:
- the **old metadata snapshot** (`..._2023-10-25_clean-v2.csv`), not a current version;
  - this version contained some errors pointed out in [encode_metadata_creation.ipynb](https://github.com/labjacquespe/EpiClass/blob/e87306bccecd3e35d9ffdbb5ddb29fcaa0800a35/src/python/epiclass/utils/notebooks/paper/encode_metadata_creation.ipynb), which re-created the metadata from scratch
  - this version did not include RNA-Seq or WGBS data

In [ ]:
EPIATLAS_16_CT = [
    ct.lower()
    for ct in [
        "T cell",
        "neutrophil",
        "brain",
        "monocyte",
        "lymphocyte of B lineage",
        "myeloid cell",
        "venous blood",
        "macrophage",
        "mesoderm-derived structure",
        "endoderm-derived structure",
        "colon",
        "connective tissue cell",
        "hepatocyte",
        "mammary gland epithelial cell",
        "muscle organ",
        "extraembryonic cell",
    ]
]

In [21]:
encode_metadata_dir = (
    Path.home() / "Projects/epiclass/output/paper/data/metadata/encode/old_meta"
)

In [39]:
meta_df = pd.read_csv(
    encode_metadata_dir / "encode_metadata_2023-10-25_clean-v2.csv", low_memory=False
)
print(meta_df.shape)
display(meta_df.head())

(9619, 183)


,md5sum,File format,File type,Output type,File assembly,Assay,Donor(s),Biosample term id,Biosample term name_x,Biosample organism,...,biosample_treatment_duration,biosample_treatment_duration_units,biosample_modification_site_target_organism,biosample_modification_site_introduced_gene_organism,replicates,cellular_component,files_replace,donor_sex,cancer_status,experiment_accession
0,ENCFF072UFE,bigWig,bigWig,signal p-value,GRCh38,H3K4me2,/human-donors/ENCDO000AAZ/,EFO:0001086,A549,Homo sapiens,...,unknown,unknown,unknown,unknown,unknown,unknown,unknown,male,cancer,ENCSR918VQU
1,ENCFF114UFW,bigWig,bigWig,signal p-value,GRCh38,KDM5A,/human-donors/ENCDO000AAZ/,EFO:0001086,A549,Homo sapiens,...,unknown,unknown,unknown,unknown,unknown,unknown,unknown,male,cancer,ENCSR933MHJ
2,ENCFF604SWJ,bigWig,bigWig,signal p-value,GRCh38,H3K4me3,/human-donors/ENCDO000AAB/,EFO:0002791,HeLa-S3,Homo sapiens,...,unknown,unknown,unknown,unknown,unknown,unknown,unknown,female,cancer,ENCSR000AOF
3,ENCFF562UDF,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,...,48,hour,unknown,unknown,/replicates/7cdba585-65b5-439c-bb03-463c8ae8da40/,unknown,"ENCFF044QFZ,ENCFF562UDF,ENCFF778HQR,ENCFF954SAX",unknown,unknown,ENCSR086GBI
4,ENCFF654CFQ,bigWig,bigWig,signal p-value,GRCh38,H3K27ac,/human-donors/ENCDO640RUC/,UBERON:0006483,middle frontal area 46,Homo sapiens,...,unknown,unknown,unknown,unknown,unknown,unknown,unknown,female,non-cancer,ENCSR004YQD


In [23]:
for column in meta_df.columns:
    print(column)

md5sum
File format
File type
Output type
File assembly
Assay
Donor(s)
Biosample term id
Biosample term name_x
Biosample organism
Biosample treatments
Biosample treatments amount
Biosample treatments duration
Biosample genetic modifications methods
Biosample genetic modifications categories
Biosample genetic modifications targets
Biosample genetic modifications gene targets
Biosample genetic modifications site coordinates
Biosample genetic modifications zygosity
Experiment target
Library made from
Library lysis method
Library crosslinking method
Experiment date released
Library fragmentation method
Library size range
Biological replicate(s)
Technical replicate(s)
Derived from
Size
Lab_x
md5sum_encode
File download URL
File Status
s3_uri
Azure URL
File analysis title
File analysis status
Audit WARNING
Audit NOT_COMPLIANT
Audit ERROR
assay_epiclass
track_type
uuid
ID
Assay name
Assay title
Target
Target of assay
Target gene symbol
biosample_summary
Biosample term name_y
Dbxrefs
Descriptio

Drop ENCODE entries linked to IHEC reference epigenomes, so the comparison is against independent samples.

In [24]:
# exclude epiatlas overlap
meta_df = meta_df[~meta_df["Related series"].str.contains("reference-epigenomes")]
print(meta_df.shape)

(7283, 183)


Attach harmonized cell-type annotations

Map each `biosample_term_id` to its harmonized ontology intermediate term via the EpiATLAS CURIE table, so ENCODE samples can be checked against the classifier's classes.

In [ ]:
curie_def_df = pd.read_csv(
    encode_metadata_dir / "EpiAtlas_list-curie_term_HSOI.tsv",
    sep="\t",
    names=["biosample_term_id", "biosample_term_name", CELL_TYPE],
)
display(curie_def_df.head())

,biosample_term_id,biosample_term_name,harmonized_sample_ontology_intermediate
0,CL:0000019,sperm,germ line cell
1,CL:0000037,hematopoietic stem cell,hematopoietic cell
2,CL:0000038,erythroid progenitor cell,hematopoietic cell
3,CL:0000041,mature eosinophil,eosinophil
4,CL:0000047,neural stem cell,neural cell


In [ ]:
new_df = pd.merge(
    left=meta_df,
    right=curie_def_df[["biosample_term_id", CELL_TYPE]],
    on="biosample_term_id",
    how="left",
)
print(new_df.shape)

(7283, 184)


In [38]:
# Uniformize labels
new_df[CELL_TYPE] = new_df[CELL_TYPE].str.lower()

## The error mechanism: restricting "overlap" to the 16 classifier classes

`EPIATLAS_16_CT` is the set of harmonized intermediate cell-type classes the Biospecimen classifier predicts, **not** the full set of EpiATLAS ontology terms. Defining overlap as "sample's cell type ∈ these 16 classes" answers a different question than "sample shares an ontology term with any EpiATLAS sample":

- it collapses to 16 coarse classes instead of the full term set, and
- a sample sharing a term *outside* the 16 counts as non-overlapping here.

This is the definition that yields ~10%; treating it as general biospecimen overlap is
the error corrected in the revision.

In [30]:
new_df[ASSAY].value_counts(dropna=False)

assay_epiclass
non-core    3200
input       1981
h3k4me3      380
h3k27ac      367
CTCF         309
h3k27me3     296
h3k4me1      260
h3k36me3     247
h3k9me3      243
Name: count, dtype: int64

In [ ]:
core_df = new_df[~new_df[ASSAY].isin(["non-core", "CTCF"])]
print(core_df[ASSAY].value_counts(dropna=False))

N = core_df[CELL_TYPE].isin(EPIATLAS_16_CT).sum()
print(N, core_df.shape[0], f"{N/len(core_df):.1%}")

assay_epiclass
input       1981
h3k4me3      380
h3k27ac      367
h3k27me3     296
h3k4me1      260
h3k36me3     247
h3k9me3      243
Name: count, dtype: int64
388 3774 10.3%
